In [1]:
# Determining how to apply actions to Go2 joints
# Reference
# https://genesis-world.readthedocs.io/en/latest/user_guide/getting_started/control_your_robot.html#joint-control
# https://github.com/Genesis-Embodied-AI/Genesis/tree/main/examples/locomotion

# Importing libraries
import genesis as gs

#Initializes Genesis with the CPU backend.
gs.init(backend=gs.gpu)

#Create a Scene
scene = gs.Scene(show_viewer=True)

#Adds a flat ground plane to the scene.
plane = scene.add_entity(gs.morphs.Plane())

#Integrate the Go2 Robot xml.
robot = gs.morphs.MJCF(file="xml/Unitree_Go2/go2.xml")

#Add an entity to the scene.
Go2 = scene.add_entity(robot)

#Builds the scene.
scene.build()

[Genesis] [16:46:26] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [16:46:26] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [16:46:26] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [16:46:27] [INFO] Running on [NVIDIA GeForce RTX 3060 Laptop GPU] with backend gs.cuda. Device memory: 6.00 GB.
[Genesis] [16:46:27] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [16:46:28] [INFO] Scene <249d6bb> created.
[Genesis] [16:46:28] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <9788623>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [16:46:29] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <6061d4a>, morph: <gs.morphs.MJCF(file='c:\Users\anich\OneDrive\Desktop\total_robotics\genesis_AI_sims\Unitree_Go2\xml\Unitree_Go2\go2.xml')>, material: <gs.materials.Rigid>.
[Genesis] [16:46:30] [WARNING] (MJCF) Friction loss at DoF-level not supported.


In [2]:
import numpy as np

# Gait frequency (Hz) and amplitude (radians)
freq = 1.0  # 1 cycle per second
amp = 0.5   # 0.5 rad swing

# Time step size (should match Genesis default)
dt = 0.01
t = 0.0

# Ordered joint list for a trot gait (same on each side, but out of phase)
joint_names = [
    'FL_hip_joint', 'FL_thigh_joint', 'FL_calf_joint',
    'FR_hip_joint', 'FR_thigh_joint', 'FR_calf_joint',
    'RL_hip_joint', 'RL_thigh_joint', 'RL_calf_joint',
    'RR_hip_joint', 'RR_thigh_joint', 'RR_calf_joint'
]

# Map joint name → dof index (you should have this already)
motor_map = {
    name: Go2.get_joint(name).dof_idx_local
    for name in joint_names
}


[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [16:59:52] [WARNING] This property is deprecated and

In [3]:
motor_map

{'FL_hip_joint': [0, 1, 2, 3, 4, 5],
 'FL_thigh_joint': 10,
 'FL_calf_joint': 14,
 'FR_hip_joint': 7,
 'FR_thigh_joint': 11,
 'FR_calf_joint': 15,
 'RL_hip_joint': 8,
 'RL_thigh_joint': 12,
 'RL_calf_joint': 16,
 'RR_hip_joint': 9,
 'RR_thigh_joint': 13,
 'RR_calf_joint': 17}

In [14]:
motor_map = {
    'FL_hip_joint': 6,   # Use the correct single DoF for actuation (NOT the first 6)
    'FL_thigh_joint': 10,
    'FL_calf_joint': 14,
    'FR_hip_joint': 7,
    'FR_thigh_joint': 11,
    'FR_calf_joint': 15,
    'RL_hip_joint': 8,
    'RL_thigh_joint': 12,
    'RL_calf_joint': 16,
    'RR_hip_joint': 9,
    'RR_thigh_joint': 13,
    'RR_calf_joint': 17,
}
motor_dofs = list(motor_map.values())  # ✅ Now 12 items, matching target_positions


In [15]:
motor_dofs

[6, 10, 14, 7, 11, 15, 8, 12, 16, 9, 13, 17]

In [16]:
Go2.set_dofs_kp(kp=np.array([3000] * len(motor_dofs)), dofs_idx_local=motor_dofs)
Go2.set_dofs_kv(kv=np.array([300] * len(motor_dofs)), dofs_idx_local=motor_dofs)


In [18]:
# Main control loop
for _ in range(1000):
    t += dt
    target_positions = []

    for joint_name in joint_order:
        phase = np.pi if ('FR' in joint_name or 'RL' in joint_name) else 0.0

        if 'hip' in joint_name:
            angle = 0.2 * np.sin(2 * np.pi * freq * t + phase)
        elif 'thigh' in joint_name:
            angle = amp * np.sin(2 * np.pi * freq * t + phase)
        else:  # calf
            angle = -amp * np.sin(2 * np.pi * freq * t + phase)

        target_positions.append(angle)

    Go2.control_dofs_position(np.array(target_positions), motor_dofs)
    scene.step()

[Genesis] [17:11:51] [INFO] Running at 0.01 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.01 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.01 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.02 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.03 FPS.
[Genesis] [17:11:51] [INFO] Running at 0.04 FPS.
[Genesis] [17:11:51]